In [3]:
import re
from collections import Counter

# ============================================================
# 1. LOAD WIKITEXT-2 DATASET
# ============================================================

# Make sure train.txt is in the same folder as this notebook

with open("train.txt", "r", encoding="utf-8") as file:
    text = file.read()

print("WikiText-2 dataset loaded successfully!")
print("Total characters:", len(text))


# ============================================================
# 2. CLEAN AND TOKENIZE THE TEXT
# ============================================================

# Convert text to lowercase
text = text.lower()

# Keep only English words
tokens = re.findall(r"[a-z]+", text)

print("Total tokens:", len(tokens))
print("First 20 tokens:")
print(tokens[:20])


# ============================================================
# 3. BUILD UNIGRAM FREQUENCY TABLE
# ============================================================

# Unigram = single word

unigram = Counter(tokens)

print("\nTop 10 Unigrams:")
print(unigram.most_common(10))


# ============================================================
# 4. BUILD BIGRAM FREQUENCY TABLE
# ============================================================

# Bigram = two consecutive words

bigram = Counter()

for i in range(len(tokens) - 1):
    word1 = tokens[i]
    word2 = tokens[i + 1]

    bigram[(word1, word2)] += 1

print("\nTop 10 Bigrams:")
print(bigram.most_common(10))


# ============================================================
# 5. BUILD TRIGRAM FREQUENCY TABLE
# ============================================================

# Trigram = three consecutive words

trigram = Counter()

for i in range(len(tokens) - 2):
    word1 = tokens[i]
    word2 = tokens[i + 1]
    word3 = tokens[i + 2]

    trigram[(word1, word2, word3)] += 1

print("\nTop 10 Trigrams:")
print(trigram.most_common(10))


# ============================================================
# 6. CALCULATE WORD PROBABILITY
# ============================================================

total_words = len(tokens)


# Unigram probability
# P(word)

def unigram_probability(word):

    return unigram[word] / total_words


# Bigram probability
# P(word2 | word1)

def bigram_probability(word1, word2):

    count_bigram = bigram[(word1, word2)]
    count_word1 = unigram[word1]

    if count_word1 == 0:
        return 0

    return count_bigram / count_word1


# Trigram probability
# P(word3 | word1, word2)

def trigram_probability(word1, word2, word3):

    count_trigram = trigram[(word1, word2, word3)]
    count_bigram = bigram[(word1, word2)]

    if count_bigram == 0:
        return 0

    return count_trigram / count_bigram


# ============================================================
# 7. FIND NEXT WORD CANDIDATES
# ============================================================

def predict_next_words(sentence, top_n=5):

    # Clean and tokenize user input
    words = re.findall(r"[a-z]+", sentence.lower())

    if len(words) == 0:
        return []

    candidates = []


    # --------------------------------------------------------
    # TRIGRAM MODEL
    # --------------------------------------------------------

    if len(words) >= 2:

        word1 = words[-2]
        word2 = words[-1]

        for word in unigram:

            count = trigram[(word1, word2, word)]

            if count > 0:

                probability = (
                    count / bigram[(word1, word2)]
                )

                candidates.append(
                    (word, probability)
                )


    # --------------------------------------------------------
    # BIGRAM MODEL
    # --------------------------------------------------------

    if len(candidates) == 0:

        last_word = words[-1]

        for word in unigram:

            count = bigram[(last_word, word)]

            if count > 0:

                probability = (
                    count / unigram[last_word]
                )

                candidates.append(
                    (word, probability)
                )


    # --------------------------------------------------------
    # UNIGRAM FALLBACK
    # --------------------------------------------------------

    if len(candidates) == 0:

        candidates = [
            (word, count / total_words)
            for word, count in unigram.most_common()
        ]


    # --------------------------------------------------------
    # RANK CANDIDATES
    # --------------------------------------------------------

    candidates.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return candidates[:top_n]


# ============================================================
# 8. TEST THE PREDICTOR
# ============================================================

print("\n======================================")
print("     NEXT WORD PREDICTION")
print("======================================")


sentence = input(
    "\nEnter a sentence or partial sentence: "
)


# Number of predictions
top_n = 5


predictions = predict_next_words(
    sentence,
    top_n
)


# ============================================================
# 9. DISPLAY TOP PREDICTIONS
# ============================================================

print("\nTop predicted next words:")

if len(predictions) == 0:

    print("No predictions found.")

else:

    for i, (word, probability) in enumerate(
        predictions,
        start=1
    ):

        print(
            f"{i}. {word} - {probability:.3f}"
        )

WikiText-2 dataset loaded successfully!
Total characters: 10810591
Total tokens: 1694562
First 20 tokens:
['york', 'city', 'f', 'c', 'season', 'the', 'season', 'was', 'the', 'unk', 'season', 'of', 'competitive', 'association', 'football', 'and', 'th', 'season', 'in', 'the']

Top 10 Unigrams:
[('the', 130768), ('of', 57030), ('unk', 54625), ('and', 50735), ('in', 45015), ('to', 39521), ('a', 36708), ('was', 21008), ('s', 16177), ('on', 15151)]

Top 10 Bigrams:
[(('of', 'the'), 17473), (('in', 'the'), 12772), (('unk', 'unk'), 6158), (('to', 'the'), 6079), (('the', 'unk'), 4582), (('on', 'the'), 4523), (('and', 'the'), 4456), (('unk', 'and'), 4357), (('for', 'the'), 3741), (('at', 'the'), 3239)]

Top 10 Trigrams:
[(('unk', 'unk', 'unk'), 1393), (('one', 'of', 'the'), 869), (('unk', 'and', 'unk'), 851), (('the', 'united', 'states'), 673), (('unk', 'unk', 'and'), 632), (('of', 'the', 'unk'), 616), (('as', 'well', 'as'), 605), (('unk', 'of', 'the'), 582), (('part', 'of', 'the'), 534), (('unk


Enter a sentence or partial sentence:  Machine learning is



Top predicted next words:
1. a - 0.122
2. the - 0.079
3. not - 0.030
4. also - 0.026
5. an - 0.023
